In [4]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import os
from rag_helper import RAGBase

In [5]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    id_field="doc_id",      # <-- use the existing id so if it's re-run, it will update existing records instead of creating duplicates
    db_path='faq.db'
)

In [6]:
index.search("how do i join a course")

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.

In [7]:
openai_client = OpenAI(api_key=os.getenv("OPENROUTER_API_KEY"),
                       base_url="https://openrouter.ai/api/v1"
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)




Yes, you can join the course now. However, if you want to receive a certificate, you'll need to ensure you submit your project while the course is still accepting submissions. Certificates are only awarded to those who complete the course with a "live" cohort, as peer-reviewing 3 capstone projects is required, which


In [8]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [9]:
assistant.rag("How do I get a certificate?")

''

In [10]:
search_results = index.search("How do I run ollama locally?")

In [11]:
search_results

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [12]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [13]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [14]:
messages = [{ "role": "user", "content": "how get certified?" }]

In [15]:
response = openai_client.responses.create(
    model="poolside/laguna-m.1:free",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseReasoningItem(id='rs_tmp_o2lt5dz2ul', summary=[], type='reasoning', content=[Content(text='\nI need to search the FAQ to find information about getting certified, as this seems like a common question that would be documented in the course resources.\n', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ResponseOutputMessage(id='msg_tmp_tker07752ys', content=[ResponseOutputText(annotations=[], text='\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None),
 ResponseFunctionToolCall(arguments='{"query": "how get certified"}', call_id='chatcmpl-tool-bf677784939c42479916667f7199e9ca', name='search', type='function_call', id='fc_tmp_wf8htfdahi', namespace=None, status='completed')]

In [16]:
call = response.output

In [17]:
call

[ResponseReasoningItem(id='rs_tmp_o2lt5dz2ul', summary=[], type='reasoning', content=[Content(text='\nI need to search the FAQ to find information about getting certified, as this seems like a common question that would be documented in the course resources.\n', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ResponseOutputMessage(id='msg_tmp_tker07752ys', content=[ResponseOutputText(annotations=[], text='\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None),
 ResponseFunctionToolCall(arguments='{"query": "how get certified"}', call_id='chatcmpl-tool-bf677784939c42479916667f7199e9ca', name='search', type='function_call', id='fc_tmp_wf8htfdahi', namespace=None, status='completed')]

In [18]:
import json

call = response.output[2]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [19]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [20]:
messages 

[{'role': 'user', 'content': 'how get certified?'},
 ResponseReasoningItem(id='rs_tmp_o2lt5dz2ul', summary=[], type='reasoning', content=[Content(text='\nI need to search the FAQ to find information about getting certified, as this seems like a common question that would be documented in the course resources.\n', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ResponseOutputMessage(id='msg_tmp_tker07752ys', content=[ResponseOutputText(annotations=[], text='\n\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None),
 ResponseFunctionToolCall(arguments='{"query": "how get certified"}', call_id='chatcmpl-tool-bf677784939c42479916667f7199e9ca', name='search', type='function_call', id='fc_tmp_wf8htfdahi', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'chatcmpl-tool-bf677784939c42479916667f7199e9ca',
  'output': '[\n  {\n    "id": "9f689c185f",\n    "course": "llm-z

In [21]:
response = openai_client.responses.create(
    model="poolside/laguna-m.1:free",
    input=messages,
    tools=[search_tool],
)

response.output_text

'\nBased on the search results, here\'s what I found about certification requirements for the LLM Zoomcamp course:\n\n1. You need to **pass the Capstone project** to receive a certificate.\n2. The course must be completed with a **"live" cohort**. Self-paced participation alone does not qualify for certification.\n3. After submitting your Capstone project, you must also complete **peer review of 3 other capstone projects**, which can only be done while the course is actively running.\n\nHowever, the search did not return specific details about the complete certification process, application procedures, or deadlines. You might want to check the official course materials or contact the course administrators directly for more comprehensive instructions on how to get certified.\n'

In [22]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(1364, 260)

In [23]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [25]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [27]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="poolside/laguna-m.1:free",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

ASSISTANT:



function_call: search {"query": "join course enrollment register late registration late joining"}


In [29]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="poolside/laguna-m.1:free",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:

Great news! **Yes, you can still join the course!** 

According to the FAQ, while you can join at any time, there's an important consideration if you want to receive a certificate: you need to submit your project while the course is still accepting submissions (meaning while it's in session with the live cohort).

A few other helpful details:
- You don't need to wait for a confirmation email - you're automatically accepted
- Registration is mainly to help gauge interest before the start date
- You can start learning and submitting homework immediately (as long as the submission form is open)
- The course materials (videos, GitHub repository, and documentation) are all available for you to begin

If you'd like to know when the next offering of this course will be (in case you're asking about a future cohort), the next one is scheduled for **Summer 2027**.

Would you like to know more about how to get started with the coursework, or do you have questions about

In [30]:
def agent_loop(instructions, question, model="poolside/laguna-m.1:free") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [31]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
ASSISTANT:

I'll help you find information about running Olama locally. Let me search the FAQ database for relevant information.


function_call: search {"query": "Olama locally"}
iteration #2...
ASSISTANT:

Let me search more specifically for running Ollama locally:


function_call: search {"query": "Ollama run locally"}
iteration #3...
ASSISTANT:



function_call: search {"query": "Ollama"}
iteration #4...
ASSISTANT:



function_call: search {"query": "local LLM"}
iteration #5...
ASSISTANT:



function_call: search {"query": "run model locally"}
iteration #6...
ASSISTANT:



function_call: search {"query": "Ollama models"}
iteration #7...
ASSISTANT:

Based on the FAQ information, here's how to run **Ollama** locally:

## Installing Ollama

First, install Ollama based on your operating system:

- **macOS**: Download the `.pkg` from [https://ollama.com/download](https://ollama.com/download) and install it
- **Windows**: Download the `.msi` from [https://ollama.com/downl

'\nBased on the FAQ information, here\'s how to run **Ollama** locally:\n\n## Installing Ollama\n\nFirst, install Ollama based on your operating system:\n\n- **macOS**: Download the `.pkg` from [https://ollama.com/download](https://ollama.com/download) and install it\n- **Windows**: Download the `.msi` from [https://ollama.com/download](https://ollama.com/download) and install it\n- **Linux**: Run this command in your terminal:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\n## Running Ollama\n\nAfter installation, you can start using Ollama immediately. In your terminal, run:\n\n```bash\nollama run llama3\n```\n\nThis will:\n- Download the LLaMA 3 model (~4GB)\n- Start the model locally on your machine\n- Open a chat-like interface where you can type questions\n\n## Testing the Local Server\n\nTo verify Ollama is running correctly, test the local server with:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response like: `{"models": [...]}`\n

In [32]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
ASSISTANT:

Let me check the course FAQ for information about enrollment and joining procedures.


function_call: search {"query": "enroll enrollment join registration late join deadline"}
iteration #2...
ASSISTANT:

Let me also check for information about deadlines and timing for new participants.


function_call: search {"query": "deadline submission homework project late registration timing"}
iteration #3...
ASSISTANT:



function_call: search {"query": "start course access materials homework submission"}
iteration #4...
ASSISTANT:

Based on the course FAQ, here's what you need to know about joining the course:

## **Yes, you can still join!** 🎉

According to the FAQ, the answer to your exact question is: **"Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions."**

### Here's how to get started:

1. **Start learning immediately** - You can begin whenever you want since the videos and GitHub materials a

'\nBased on the course FAQ, here\'s what you need to know about joining the course:\n\n## **Yes, you can still join!** 🎉\n\nAccording to the FAQ, the answer to your exact question is: **"Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions."**\n\n### Here\'s how to get started:\n\n1. **Start learning immediately** - You can begin whenever you want since the videos and GitHub materials are available\n2. **Check the course materials**:\n   - [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n   - [Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)\n   - [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n3. **Find deadlines** on the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/)\n4. **Follow the weekly workflow**:\n   - Watch lesson videos\n   - Work through lesson notebooks/code\n   - Read homework instructio

In [33]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
ASSISTANT:



function_call: search {"query": "join course enrollment late registration"}
iteration #2...
ASSISTANT:

Yes, you can still join the LLM Zoomcamp course! According to the FAQ, you're welcome to start learning regardless of when you discovered it.

**However, there's an important consideration for certification:**
- If you want to receive a certificate, you need to submit your project while the course is still accepting submissions
- For certificate eligibility, you must finish with a "live" cohort and complete peer-review requirements (reviewing 3 capstone projects)

**To get started:**
1. You don't need to wait for a confirmation email - you're automatically accepted
2. Begin with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), [logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)
3. Check the [course management platform](https://cour

'\nYes, you can still join the LLM Zoomcamp course! According to the FAQ, you\'re welcome to start learning regardless of when you discovered it.\n\n**However, there\'s an important consideration for certification:**\n- If you want to receive a certificate, you need to submit your project while the course is still accepting submissions\n- For certificate eligibility, you must finish with a "live" cohort and complete peer-review requirements (reviewing 3 capstone projects)\n\n**To get started:**\n1. You don\'t need to wait for a confirmation email - you\'re automatically accepted\n2. Begin with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), [logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n3. Check the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for deadlines and schedule\n4. Follow the typical workflow: watch videos → work

In [34]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
ASSISTANT:



function_call: search {"query": "queen gambit"}
iteration #2...
ASSISTANT:



function_call: search {"query": "queen's gambit"}
iteration #3...
ASSISTANT:

It looks like the FAQ database doesn't contain specific information about "Queen's Gambit." Let me try searching more broadly or with different terms.


function_call: search {"query": "chess"}
iteration #4...
ASSISTANT:



function_call: search {"query": "gambit"}
iteration #5...
ASSISTANT:

Based on my search of the course FAQ database, there doesn't appear to be specific information about "Queen's Gambit" in the course materials. However, I can provide you with general information about what the Queen's Gambit is:

## Queen's Gambit (Chess)

The Queen's Gambit is a classic chess opening that begins with the moves:

1. **d4 d5**
2. **c4**

This is called a "gambit" because White offers a pawn (the c4 pawn) to Black. It's one of the oldest known chess openings, with recorded games from the 15th century

'\nBased on my search of the course FAQ database, there doesn\'t appear to be specific information about "Queen\'s Gambit" in the course materials. However, I can provide you with general information about what the Queen\'s Gambit is:\n\n## Queen\'s Gambit (Chess)\n\nThe Queen\'s Gambit is a classic chess opening that begins with the moves:\n\n1. **d4 d5**\n2. **c4**\n\nThis is called a "gambit" because White offers a pawn (the c4 pawn) to Black. It\'s one of the oldest known chess openings, with recorded games from the 15th century.\n\n### Key Points:\n- **Strategic goal**: White aims to control the center and gain rapid piece development\n- **The "gambit"**: White offers the c4 pawn to lure Black\'s pieces away from the center\n- **Accepted/Declined**: Black can accept the pawn (Queen\'s Gambit Accepted) or decline it (Queen\'s Gambit Declined)\n- **Popularity**: It was the most played opening in top-level chess until recently, and featured prominently in the Netflix series "The Quee

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
ASSISTANT:

I'll search the course FAQ database to see if "queen gambit" is mentioned in any course-related context.


function_call: search {"query": "queen gambit"}
iteration #2...
ASSISTANT:

Let me search more broadly to see if there's any chess-related content in the course:


function_call: search {"query": "chess"}
iteration #3...
ASSISTANT:

Let me check if there's any gaming or entertainment content in the course:


function_call: search {"query": "gaming entertainment"}
iteration #4...
ASSISTANT:

I couldn't find any information about "queen gambit" or related chess/gaming content in the course FAQ database. This appears to be an off-topic question.

As a course teaching assistant, I can only answer questions that are related to the course content or its logistics as documented in the FAQ database. "The Queen's Gambit" is a popular Netflix series about chess, which isn't related to the course material based on my search.

Is there anything else about the cours

'\nI couldn\'t find any information about "queen gambit" or related chess/gaming content in the course FAQ database. This appears to be an off-topic question.\n\nAs a course teaching assistant, I can only answer questions that are related to the course content or its logistics as documented in the FAQ database. "The Queen\'s Gambit" is a popular Netflix series about chess, which isn\'t related to the course material based on my search.\n\nIs there anything else about the course content or logistics that you\'d like me to help you with?\n'